# SupportIQ — Stage 1.1: Load & Inspect Raw Data

> **Goal:** Ingest raw Bitext customer support dataset, record provenance in `METADATA.json`, and verify columns and samples.


### 1. Setup & Environment

In [1]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

import polars as pl

from supportiq.data.load import ingest_raw_dataset

print(f"Polars: {pl.__version__} | Project Root: {project_root}")

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Polars: 1.44.2 | Project Root: /home/dvinix/Projects/supportiq


### 2. Ingest Dataset & Save Raw Artifacts

In [2]:
# Ingest dataset from Hugging Face and write raw files + METADATA.json
df, metadata = ingest_raw_dataset(output_dir=project_root / "data/raw")

print(f"Dataset:  {metadata.dataset_name}")
print(f"Revision: {metadata.revision_sha}")
print(f"Rows:     {metadata.row_count:,} rows across {metadata.column_count} columns")
print(f"Columns:  {metadata.columns}")

{"timestamp": "2026-09-21T19:49:46.961506+00:00", "level": "INFO", "name": "supportiq.data.load", "message": "Connecting to Hugging Face Hub for metadata: bitext/Bitext-customer-support-llm-chatbot-training-dataset", "module": "load", "line": 78}


{"timestamp": "2026-09-21T19:49:47.368465+00:00", "level": "INFO", "name": "supportiq.data.load", "message": "Downloading dataset split bitext/Bitext-customer-support-llm-chatbot-training-dataset (split: train, revision: 430d1a89bd93bd1fa23c16f29dd53e73f0087443)", "module": "load", "line": 85}


{"timestamp": "2026-09-21T19:49:49.887032+00:00", "level": "INFO", "name": "supportiq.data.load", "message": "Persisting raw artifacts: /home/dvinix/Projects/supportiq/data/raw/bitext_raw.parquet, /home/dvinix/Projects/supportiq/data/raw/bitext_raw.csv", "module": "load", "line": 98}


{"timestamp": "2026-09-21T19:49:49.989977+00:00", "level": "INFO", "name": "supportiq.data.load", "message": "Raw dataset ingestion complete (26872 rows, 5 columns, sha256=0c8ae53ede333a008aa821ee287c93a91d166b6f80cb81a3d665fa55d9a48782)", "module": "load", "line": 129}


Dataset:  bitext/Bitext-customer-support-llm-chatbot-training-dataset
Revision: 430d1a89bd93bd1fa23c16f29dd53e73f0087443
Rows:     26,872 rows across 5 columns
Columns:  ['flags', 'instruction', 'category', 'intent', 'response']


### 3. Check for Nulls & Verify Integrity

In [3]:
# Check null counts
null_summary = df.null_count()
print("Null count per column:")
print(null_summary)

Null count per column:
shape: (1, 5)
┌───────┬─────────────┬──────────┬────────┬──────────┐
│ flags ┆ instruction ┆ category ┆ intent ┆ response │
│ ---   ┆ ---         ┆ ---      ┆ ---    ┆ ---      │
│ u32   ┆ u32         ┆ u32      ┆ u32    ┆ u32      │
╞═══════╪═════════════╪══════════╪════════╪══════════╡
│ 0     ┆ 0           ┆ 0        ┆ 0      ┆ 0        │
└───────┴─────────────┴──────────┴────────┴──────────┘


### 4. Inspect Sample Records (5 Rows)

In [4]:
sample_rows = df.sample(n=5, seed=42)
for i, row in enumerate(sample_rows.iter_rows(named=True), start=1):
    print(f"[{i}] Category: {row['category']} | Intent: {row['intent']}")
    print(f"    User:      {row['instruction']}")
    print(f"    Assistant: {row['response'][:100]}...")
    print("-" * 70)

[1] Category: ACCOUNT | Intent: registration_problems
    User:      I have got to inform of a problem with a registration
    Assistant: Thank you for bringing this issue to our attention. We understand the frustration that can arise whe...
----------------------------------------------------------------------
[2] Category: CONTACT | Intent: contact_customer_service
    User:      help checking what hours I can call customer assistance
    Assistant: Thank you for reaching out! I completely understand your need to know the operating hours for our cu...
----------------------------------------------------------------------
[3] Category: REFUND | Intent: track_refund
    User:      need help checking if there are any updates on the refund
    Assistant: I can see that you are eager to check for any updates on your refund. Let me quickly look into the s...
----------------------------------------------------------------------
[4] Category: PAYMENT | Intent: payment_issue
    User:      i